# Level 9: NISQ Era and Hybrid Quantum-Classical Algorithms
## Qiskit Edition

In this notebook, you will:

1. Understand the **NISQ era** (Noisy Intermediate-Scale Quantum)
2. Learn about **variational algorithms** — the most practical NISQ approach
3. Implement a **Variational Quantum Eigensolver (VQE)** from scratch
4. Build a simple **QAOA** circuit for combinatorial optimization

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import SparsePauliOp, Statevector
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize

simulator = AerSimulator()
print("Ready!")

---
## 1. The NISQ Era

### What is NISQ?
- **N**oisy: Current qubits have error rates of ~0.1–1%
- **I**ntermediate-**S**cale: 50–1000+ qubits available today
- **Q**uantum: Still quantum, still useful!

### The Challenge
NISQ devices can't run Shor's or large Grover's directly — they need too many gates and too much error correction.

### The Solution: Hybrid Algorithms
Use quantum computers for what they're best at (exploring quantum state space) and classical computers for the rest:

1. **VQE** (Variational Quantum Eigensolver) → Chemistry & Materials
2. **QAOA** (Quantum Approximate Optimization) → Combinatorial optimization

---
## 2. Variational Quantum Eigensolver (VQE)

### The Idea
Find the ground state energy of a Hamiltonian $H$ by:

1. **Quantum part**: Prepare a parameterized state $|\psi(\theta)\rangle$ and measure $\langle H \rangle$
2. **Classical part**: Optimize $\theta$ to minimize $\langle \psi(\theta)|H|\psi(\theta)\rangle$

By the variational principle: $\langle \psi(\theta)|H|\psi(\theta)\rangle \geq E_0$ always!

### Simple Example: Finding the ground state of a 2-qubit Hamiltonian

$$H = -1.0 \cdot ZZ + 0.5 \cdot XI + 0.5 \cdot IX$$

In [ ]:
# Define the Hamiltonian
hamiltonian = SparsePauliOp.from_list([
    ('ZZ', -1.0),
    ('XI', 0.5),
    ('IX', 0.5)
])
print("Hamiltonian:")
print(hamiltonian)

# Find exact ground state energy for comparison
eigenvalues = np.linalg.eigvalsh(hamiltonian.to_matrix().toarray())
exact_ground = min(eigenvalues)
print(f"\nExact ground state energy: {exact_ground:.6f}")

In [ ]:
def ansatz(params):
    """Parameterized quantum circuit (ansatz)."""
    qc = QuantumCircuit(2)
    
    # Layer 1: Single-qubit rotations
    qc.ry(params[0], 0)
    qc.ry(params[1], 1)
    
    # Entangling layer
    qc.cx(0, 1)
    
    # Layer 2: More rotations
    qc.ry(params[2], 0)
    qc.ry(params[3], 1)
    
    return qc

# Show the ansatz
example_params = [0.5, 0.5, 0.5, 0.5]
ansatz(example_params).draw('mpl')

In [ ]:
def compute_expectation(params):
    """Compute <psi(params)|H|psi(params)> using statevector simulation."""
    qc = ansatz(params)
    sv = Statevector.from_instruction(qc)
    expectation = sv.expectation_value(hamiltonian).real
    return expectation

# Test with random parameters
test_params = np.random.random(4) * 2 * np.pi
energy = compute_expectation(test_params)
print(f"Random parameters energy: {energy:.6f}")
print(f"Exact ground state:       {exact_ground:.6f}")

In [ ]:
# VQE optimization loop
energy_history = []

def objective(params):
    energy = compute_expectation(params)
    energy_history.append(energy)
    return energy

# Start with random initial parameters
initial_params = np.random.random(4) * 2 * np.pi

# Optimize using classical optimizer (COBYLA)
result = minimize(objective, initial_params, method='COBYLA', 
                  options={'maxiter': 200})

print(f"VQE Result:")
print(f"  Optimized energy: {result.fun:.6f}")
print(f"  Exact energy:     {exact_ground:.6f}")
print(f"  Error:            {abs(result.fun - exact_ground):.6f}")
print(f"  Iterations:       {result.nfev}")

# Plot convergence
plt.figure(figsize=(10, 5))
plt.plot(energy_history, 'b-', alpha=0.7)
plt.axhline(y=exact_ground, color='r', linestyle='--', label=f'Exact: {exact_ground:.4f}')
plt.xlabel('Iteration')
plt.ylabel('Energy')
plt.title('VQE Convergence')
plt.legend()
plt.show()

---
## 3. QAOA — Quantum Approximate Optimization

QAOA finds approximate solutions to combinatorial optimization problems.

### Simple Example: MaxCut

Given a graph, partition vertices into two groups to maximize the number of edges between groups.

For a triangle graph (3 vertices, 3 edges): The answer is any partition that puts 1 vertex in one group and 2 in the other (MaxCut = 2).

In [ ]:
def qaoa_circuit(gamma, beta, edges):
    """QAOA circuit for MaxCut on a graph."""
    n_qubits = max(max(e) for e in edges) + 1
    qc = QuantumCircuit(n_qubits, n_qubits)
    
    # Initial superposition
    qc.h(range(n_qubits))
    
    # Problem unitary (cost Hamiltonian)
    for (i, j) in edges:
        qc.cx(i, j)
        qc.rz(gamma, j)
        qc.cx(i, j)
    
    # Mixer unitary
    qc.rx(2 * beta, range(n_qubits))
    
    # Measure
    qc.measure(range(n_qubits), range(n_qubits))
    
    return qc

# Triangle graph: edges 0-1, 0-2, 1-2
edges = [(0, 1), (0, 2), (1, 2)]

qc = qaoa_circuit(np.pi/4, np.pi/8, edges)
qc.draw('mpl')

In [ ]:
def maxcut_value(bitstring, edges):
    """Compute the MaxCut value for a given partition."""
    cut = 0
    for (i, j) in edges:
        if bitstring[i] != bitstring[j]:
            cut += 1
    return cut

def qaoa_objective(params, edges):
    """Expected MaxCut value for given QAOA parameters."""
    gamma, beta = params
    qc = qaoa_circuit(gamma, beta, edges)
    counts = simulator.run(qc, shots=2000).result().get_counts()
    
    total = sum(counts.values())
    avg_cut = sum(
        maxcut_value(bitstring, edges) * count / total
        for bitstring, count in counts.items()
    )
    return -avg_cut  # Minimize negative = maximize

# Optimize QAOA parameters
result = minimize(qaoa_objective, [0.5, 0.5], args=(edges,), method='COBYLA')
opt_gamma, opt_beta = result.x

print(f"Optimal QAOA parameters: γ={opt_gamma:.4f}, β={opt_beta:.4f}")
print(f"Expected MaxCut: {-result.fun:.4f} (optimal is 2)")

# Run with optimal parameters
qc = qaoa_circuit(opt_gamma, opt_beta, edges)
counts = simulator.run(qc, shots=4000).result().get_counts()

print(f"\nOutput distribution:")
for bs, count in sorted(counts.items(), key=lambda x: -x[1]):
    cut = maxcut_value(bs, edges)
    print(f"  |{bs}> → cut={cut}, count={count}")

plot_histogram(counts, title="QAOA MaxCut on Triangle")

---
## 4. Understanding Noise

NISQ devices have noise. Let's see its effect:

In [ ]:
from qiskit_aer.noise import NoiseModel, depolarizing_error

# Create a simple noise model
noise_model = NoiseModel()
# 1% depolarizing error on single-qubit gates
error_1q = depolarizing_error(0.01, 1)
# 5% depolarizing error on two-qubit gates
error_2q = depolarizing_error(0.05, 2)

noise_model.add_all_qubit_quantum_error(error_1q, ['h', 'rx', 'ry', 'rz'])
noise_model.add_all_qubit_quantum_error(error_2q, ['cx'])

noisy_sim = AerSimulator(noise_model=noise_model)

# Compare: ideal vs noisy Bell state
qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])

ideal_result = simulator.run(qc, shots=10000).result().get_counts()
noisy_result = noisy_sim.run(qc, shots=10000).result().get_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
plot_histogram(ideal_result, ax=ax1, title="Ideal Bell State")
plot_histogram(noisy_result, ax=ax2, title="Noisy Bell State (1% / 5%)")
plt.tight_layout()
plt.show()
print("Noise introduces errors — |01> and |10> appear!")

---
## 5. Key Takeaways

| Concept | Description |
|---------|-------------|
| **NISQ** | Current era: noisy, 50–1000+ qubits |
| **VQE** | Hybrid algorithm for ground state energy |
| **QAOA** | Hybrid algorithm for combinatorial optimization |
| **Ansatz** | Parameterized quantum circuit tuned classically |
| **Noise** | Main limitation of current quantum hardware |

---
## 6. Challenges

1. **Deeper ansatz**: Add more rotation + entangling layers to the VQE. Does it converge faster?
2. **4-vertex MaxCut**: Build a QAOA circuit for a 4-vertex graph. Compare with brute-force.
3. **Noise study**: Run VQE with different noise levels. At what noise rate does it fail?

In [ ]:
# Your challenge code here!

---
**Next up: [Level 10 — Quantum Error Correction](../Level_10_Error_Correction/Level_10_Qiskit.ipynb)**